> 在线讲义：[第 61 章 · 从系统方案到期限与硬件选择：32 卡还是 48 卡](https://wikieden.github.io/ai-infra-from-constraints/lessons/61-training-choice.html) · 本书《从约束推导 AI Infra》

# 第 61 章 · 从系统方案到期限与硬件选择：32 卡还是 48 卡

配套讲义：仓库里 `ai-infra/notes/lessons/61-training-choice.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



32 / 48 卡 RTX 4090 方案的完成时间、每步预算、共用网卡、Amdahl 式硬件替换与模型规模的卡数边界，取自 AII 第 10 章 §10.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
STEPS = 31790
DAY = 86400

**函数 `finish_days`**：T_finish = T0 + ⌈D/b⌉ t_s (1 + L)。

In [ ]:
def finish_days(ts: float, L: float, T0: float = 5, steps: int = STEPS) -> float:
    """T_finish = T0 + ⌈D/b⌉ t_s (1 + L)。"""
    return T0 + steps * ts * (1 + L) / DAY

**函数 `step_budget`**：小工具。

In [ ]:
def step_budget(days: float = 25, L: float = 0.0108, steps: int = STEPS) -> float:
    return days * DAY / (steps * (1 + L))

**函数 `amdahl`**：T'/T = 1 − f + f/r。

In [ ]:
def amdahl(f: float, r: float) -> float:
    """T'/T = 1 − f + f/r。"""
    return 1 - f + f / r

**函数 `cards_needed`**：p ≥ 6ND / (P η T)，返回连续值。

In [ ]:
def cards_needed(params: float, tokens: float, peak: float, eta: float, days: float) -> float:
    """p ≥ 6ND / (P η T)，返回连续值。"""
    return 6 * params * tokens / (peak * eta * days * DAY)

**函数 `days_needed`**：小工具。

In [ ]:
def days_needed(params: float, tokens: float, peak: float, eta: float, cards: int) -> float:
    return 6 * params * tokens / (peak * eta * cards * DAY)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：汇总")
for name, ts, L in (("32 卡", 78.3 + 0.08 + 0.5, 0.0102), ("48 卡", 52.2 + 0.08 + 0.5, 0.0108)):
    print(f"  {name}：每步 {ts:.1f} s，基础 {finish_days(ts, 0, 0):.1f} 天，加保存恢复和 5 天 → {finish_days(ts, L):.1f} 天")

In [ ]:
print("\n第二步：可行范围")
b = step_budget()
print(f"  每步上限 {b:.1f} s；48 卡余量 {b - 52.78:.1f} s；通信最多 {b - 52.2 - 0.5:.1f} s")
print(f"  预取全失效 12.0 s 全暴露：{52.2 + 12.0 + 0.5:.1f} s，仍可行")
left = step_budget(L=0.0102) - 0.08 - 0.5
print(f"  32 卡留给计算 {left:.1f} s，要效率 {0.4 * 78.3 / left * 100:.0f}%")
v = 385e9
print(f"  八卡共用一张 200 Gbit/s：{v / 25e9:.1f} s > {b - 52.7:.1f} s；预取失效时每步 {52.2 + 0.5 + v / 25e9:.1f} s → 超期")

In [ ]:
print("\n第三步：换硬件")
print(f"  100 ms 里通信 5 ms，带宽减半 → {100 * amdahl(0.05, 0.5):.0f} ms；通信 50 ms → {100 * amdahl(0.5, 0.5):.0f} ms")
ts30 = 52.2 * 0.4 / 0.3
print(f"  48 卡效率 40% → 30%：单卡 {ts30:.1f} s，每步 {ts30 + 0.58:.1f} s > {b:.1f}")

In [ ]:
print("\n第四步：模型规模（20T token、16,384 张卡）")
for name, peak in (("A100", 312e12), ("H100", 989e12), ("B200", 2.25e15)):
    print(f"  1T 参数 {name} 40%：{days_needed(1e12, 20e12, peak, 0.4, 16384):.0f} 天")
print(f"  B200 90 天要 {math.ceil(cards_needed(1e12, 20e12, 2.25e15, 0.4, 90)):,} 张；50% 时 {days_needed(1e12, 20e12, 2.25e15, 0.5, 16384):.0f} 天")
print("  D = 20N 时 F = 120N²：参数 5 倍，计算 25 倍")

## 3. 代码里才看得到的

1. 预取全失效那行按 52.2 + 12.0 + 0.5 = 64.7 s，原文写 64.8 s（链路时间未取整为 12.03 s）。
2. B200 的稠密 BF16 峰值用 2.25 PFLOP/s，正好复现 94 天；换别的峰值数字天数会变。
3. `step_budget` 的 25 天是 30 天扣掉 5 天预留；附加比例 1.08% 来自第 59 章。

## 4. 自己改一改

- 把 `finish_days` 的预留改成 3 天，看 32 卡要多少效率。
- 用 `amdahl` 画 48 卡方案换成 PCIe 3.0（16 GB/s）且预取失效时的每步时间。
- 用 `cards_needed` 找 B200、MFU 40% 时 90 天能训的最大模型（D = 20N）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 总工期 = 等材料 + 天数 × 每天工时 × (1 + 返工)；
> 剩下的预算摊回每天，才知道哪项还能花；
> 修路省多少，看那段路占全程多少；
> 扩建要加几倍人，看材料是不是也翻倍。